# درس ۰۲ - بررسی چارچوب Microsoft Agent

**چارچوب Microsoft Agent (MAF)** یک چارچوب یکپارچه برای ساخت عوامل هوش مصنوعی است. این چارچوب معماری تمیز و ترکیبی با چهار بلوک اصلی ارائه می‌دهد:

- **مشتری** – به نقطه پایان مدل هوش مصنوعی متصل می‌شود و ارتباط را مدیریت می‌کند
- **عامل** – یک مشتری را با دستورالعمل‌ها و تعریف ابزارها می‌پوشاند
- **ابزارها** – قابلیت‌های عامل را با توابع سفارشی که مدل می‌تواند فراخوانی کند گسترش می‌دهند
- **نشست** – تاریخچه مکالمه را برای تعاملات چندمرحله‌ای حفظ می‌کند

در این درس، ما یک **عامل رزرو سفر** می‌سازیم که با استفاده از این مفاهیم، در دسترس بودن مقصد را بررسی می‌کند.


## راه‌اندازی


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## درک معماری چارچوب عامل

چارچوب عامل مایکروسافت از معماری چندلایه پیروی می‌کند:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **مشتری** – یک `AzureAIProjectAgentProvider` به یک استقرار Azure OpenAI متصل می‌شود. این لایه مسئول احراز هویت، قالب‌بندی درخواست و تجزیه پاسخ است.
2. **عامل** – ایجاد شده از مشتری از طریق `provider.create_agent()`، عامل دسترسی به مدل را با دستورالعمل‌ها (پرامپت سیستم) و ابزارها ترکیب می‌کند.
3. **ابزارها** – توابع پایتون که با `@tool` تزئین شده‌اند و عامل می‌تواند برای انجام اقدامات یا بازیابی داده‌ها آنها را فراخوانی کند.
4. **جلسه** – یک شیء `AgentSession` (ایجاد شده از طریق `agent.create_session()`) که تاریخچه گفتگو را ذخیره می‌کند و امکان دیالوگ چند مرحله‌ای را فراهم می‌آورد که در آن عامل زمینه قبلی را به خاطر می‌سپارد.

بیایید هر لایه را گام به گام بسازیم.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## افزودن ابزارها با دکوراتور @tool

ابزارها به عامل‌ها اجازه می‌دهند فراتر از تولید متن اقدام کنند. دکوراتور `@tool` یک تابع عادی پایتون را به چیزی تبدیل می‌کند که عامل بتواند آن را فراخوانی کند.

نکات کلیدی:
- از `Annotated[type, "description"]` استفاده کنید تا مدل هر پارامتر را درک کند.
- رشته مستندسازی (docstring) به‌عنوان توضیح ابزار برای مدل نمایش داده می‌شود.
- `approval_mode="never_require"` به این معنی است که ابزار به‌طور خودکار و بدون تأیید کاربر اجرا می‌شود.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ایجاد یک عامل با ابزارها

حالا ما مشتری، دستورالعمل‌ها و ابزارها را در یک عامل ترکیب می‌کنیم. `instructions` به عنوان راهنمای سیستم عمل می‌کند — آنها شخصیت و رفتار عامل را تعریف می‌کنند.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## مکالمات چندنوبتی با جلسات

یک `AgentSession` (که از طریق `agent.create_session()` ایجاد می‌شود) تمام پیام‌های یک مکالمه را پیگیری می‌کند. با ارسال همان جلسه به هر فراخوانی `agent.run()`، عامل به کل تاریخچه مکالمه دسترسی دارد و می‌تواند به پیام‌های قبلی مراجعه کند.

ما `tools=[check_destination_availability]` را ارسال می‌کنیم تا عامل بتواند در هر نوبت از بررسی‌کننده‌ی دسترسی ما استفاده کند.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## خلاصه

در این درس چهار ستون چارچوب Microsoft Agent را بررسی کردید:

| مفهوم | آنچه آموختید |
|---------|------------------|
| **کلاینت** | `AzureAIProjectAgentProvider` به Azure OpenAI با احراز هویت مبتنی بر اعتبارنامه متصل می‌شود |
| **عامل** | `provider.create_agent()` یک اتصال مدل را همراه با دستورالعمل‌ها و نام گروه‌بندی می‌کند |
| **ابزارها** | دکوراتور `@tool` توابع پایتون را برای فراخوانی توسط عامل در دسترس قرار می‌دهد |
| **نشست** | `agent.create_session()` تاریخچه گفتگو را در چندین مرحله حفظ می‌کند |

این بلوک‌های ساختمانی با هم ترکیب می‌شوند تا عامل‌هایی بسازند که بتوانند گفتگوهای طبیعی داشته باشند، توابع خارجی را فراخوانی کنند و زمینه را حفظ کنند — پایه‌ای برای الگوهای عامل پیشرفته‌تر در درس‌های بعدی.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:  
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما برای دقت تلاش می‌کنیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است دارای خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری آن باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسان توصیه می‌شود. ما در قبال هرگونه سوءتفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
